# PepDesign-Active: ConvVAE Training on Kaggle
**Purpose:** Train ConvVAE on 52,517 food-derived peptides using Kaggle's free T4 GPU.
**Runtime:** ~8 min. **Output:** 6 files auto-downloaded.

In [ ]:
# 1. Setup: Enable GPU (Settings -> Accelerator -> T4 GPU x2)
!pip install -q pandas numpy scikit-learn torch torchvision matplotlib

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, time, os, json, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
# 2. Upload peptide CSV (click folder icon -> upload)
print('Upload merged_peptide_library.csv using the file browser (right sidebar -> Add Data -> Upload)')
print('File: D:/projects/walnut-peptide-pilot/data/merged_peptide_library.csv')

# After upload, list files to find it
import glob
csv_files = glob.glob('/kaggle/input/*/merged_peptide_library.csv') + glob.glob('*.csv')
print(f'Found CSV files: {csv_files}')
if not csv_files:
    print('No CSV found yet. Please upload the file first.')
else:
    csv_path = csv_files[0]
    print(f'Using: {csv_path}')

In [ ]:
# 3. Load + preprocess peptides
csv_files = glob.glob('/kaggle/input/*/merged_peptide_library.csv') + glob.glob('*.csv')
if not csv_files:
    raise FileNotFoundError('Upload merged_peptide_library.csv first (Step 2)')

df = pd.read_csv(csv_files[0])
unique = df.drop_duplicates(subset=['peptide'])
unique = unique[unique['length'] >= 2]
peptides_all = unique['peptide'].tolist()

random.seed(42)
peptides = random.sample(peptides_all, min(52517, len(peptides_all)))

print(f'Loaded: {len(peptides)} peptides')
print(f'Length: mean={np.mean([len(p) for p in peptides]):.1f}, '
      f'range=[{min(len(p) for p in peptides)}, {max(len(p) for p in peptides)}]')
print(f'Sample: {peptides[:5]}')

In [ ]:
# 4. One-hot encoding
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}
MAX_LEN, N_AA = 20, 20

def peptide_to_onehot(seq):
    mat = np.zeros((MAX_LEN, N_AA), dtype=np.float32)
    for i, aa in enumerate(seq[:MAX_LEN]):
        if aa in AA_TO_IDX:
            mat[i, AA_TO_IDX[aa]] = 1.0
    return mat

print('One-hot encoding...')
t0 = time.time()
onehot = np.stack([peptide_to_onehot(p) for p in peptides])
onehot = np.transpose(onehot, (0, 2, 1))
print(f'Done: {onehot.shape} in {time.time()-t0:.1f}s')

In [ ]:
# 5. ConvVAE Model
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc_conv1 = nn.Conv1d(N_AA, 32, 3, padding=1)
        self.enc_conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.enc_bn1 = nn.BatchNorm1d(32)
        self.enc_bn2 = nn.BatchNorm1d(64)
        self.enc_pool = nn.AdaptiveAvgPool1d(5)
        self.enc_fc1 = nn.Linear(64 * 5, 128)
        self.enc_fc_mu = nn.Linear(128, latent_dim)
        self.enc_fc_logvar = nn.Linear(128, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, 128)
        self.dec_fc2 = nn.Linear(128, 64 * 5)
        self.dec_deconv1 = nn.ConvTranspose1d(64, 32, 3, padding=1)
        self.dec_bn1 = nn.BatchNorm1d(32)
        self.dec_deconv2 = nn.ConvTranspose1d(32, N_AA, 3, padding=1)

    def encode(self, x):
        h = F.relu(self.enc_bn1(self.enc_conv1(x)))
        h = F.relu(self.enc_bn2(self.enc_conv2(h)))
        h = self.enc_pool(h)
        h = h.view(h.size(0), -1)
        h = F.relu(self.enc_fc1(h))
        return self.enc_fc_mu(h), self.enc_fc_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        h = h.view(h.size(0), 64, 5)
        h = F.relu(self.dec_bn1(self.dec_deconv1(h)))
        return self.dec_deconv2(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def decode_latent(self, z_np):
        self.eval()
        with torch.no_grad():
            z = torch.tensor(z_np, dtype=torch.float32).unsqueeze(0).to(device)
            logits = self.decode(z)
            idx = logits.squeeze(0).argmax(dim=0).cpu().numpy()
            return ''.join(AA_LIST[i] for i in idx if i < len(AA_LIST))

model = ConvVAE().to(device)
print(f'ConvVAE params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 6. Train ConvVAE (80 epochs, ~5 min on T4)
def vae_loss(recon_x, x, mu, logvar, beta=0.3):
    """Beta-VAE: reconstruction CE (masked) + KL."""
    B, C, L = x.shape
    target = x.argmax(dim=1)
    mask = (x.sum(dim=1) > 0).float()
    recon = recon_x.permute(0, 2, 1).reshape(-1, C)
    target = target.reshape(-1)
    mask = mask.reshape(-1)
    ce = F.cross_entropy(recon, target, reduction='none')
    bce = (ce * mask).sum() / (mask.sum() + 1e-8)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    return bce + beta * kl

X = torch.tensor(onehot, dtype=torch.float32)
loader = DataLoader(TensorDataset(X), batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f'Training: {len(peptides)} peptides, {len(loader)} batches/epoch, 80 epochs')
losses, t0 = [], time.time()

for epoch in range(80):
    model.train()
    beta = min(0.3, 0.3 * (epoch + 1) / 40)
    ep_loss = 0.0
    for (batch,) in loader:
        batch = batch.to(device)
        opt.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar, beta)
        loss.backward()
        opt.step()
        ep_loss += loss.item()
    losses.append(ep_loss / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/80  loss={losses[-1]:.4f}  beta={beta:.2f}  {time.time()-t0:.0f}s')

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'Loss: {losses[0]:.2f} -> {losses[-1]:.2f}')

In [ ]:
# 7. Reconstruction accuracy
model.eval()
test_n = min(5000, len(peptides))
test_t = torch.tensor(onehot[:test_n], dtype=torch.float32).to(device)
with torch.no_grad():
    recon_idx = model(test_t)[0].argmax(dim=1).cpu().numpy()

ids = []
for i in range(test_n):
    orig, recon = peptides[i], ''.join(AA_LIST[j] for j in recon_idx[i] if j < len(AA_LIST))
    if len(orig) > 0:
        ids.append(sum(1 for k in range(min(len(orig), len(recon))) if orig[k] == recon[k]) / len(orig))
print(f'Reconstruction accuracy: {np.mean(ids):.3f} ({np.mean(ids)*100:.1f}%)')
print(f'Median: {np.median(ids):.3f}')

In [ ]:
# 8. Extract all latent vectors
print('Extracting latent vectors...')
model.eval()
latents = []
with torch.no_grad():
    for i in range(0, len(onehot), 1024):
        batch = torch.tensor(onehot[i:i+1024], dtype=torch.float32).to(device)
        mu, _ = model.encode(batch)
        latents.append(mu.cpu().numpy())
latents = np.vstack(latents).astype(np.float32)
print(f'Latents: {latents.shape}')

from sklearn.decomposition import PCA
pca_lat = PCA(n_components=64).fit(latents)
var = pca_lat.explained_variance_ratio_.sum()
print(f'PCA explained variance of VAE latent: {var:.3f}')

In [ ]:
# 9. Figures
import matplotlib.pyplot as plt

# Fig S1: Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(losses)+1), losses, 'b-', lw=1.5)
ax.axvline(40, color='gray', ls='--', alpha=0.5, label='Beta anneal end')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('ConvVAE Training Loss (Fig. S1)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('fig_s1_vae_loss.png', dpi=300); plt.show()

# Fig S2: Latent PCA
n_vis = min(3000, len(latents))
idx_vis = np.random.choice(len(latents), n_vis, replace=False)
lat_2d = PCA(n_components=2).fit_transform(latents[idx_vis])
KD = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}
colors = [np.mean([KD.get(aa,0) for aa in peptides[i]]) for i in idx_vis]
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(lat_2d[:,0], lat_2d[:,1], c=colors, cmap='RdYlBu', s=3, alpha=0.5)
plt.colorbar(sc, label='Hydrophobicity')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('VAE Latent Space (Fig. S2)')
plt.tight_layout(); plt.savefig('fig_s2_latent_pca.png', dpi=300); plt.show()

In [ ]:
# 10. Save + download all outputs
torch.save(model.state_dict(), 'conv_vae_weights.pt')
np.savez_compressed('vae_latents.npz', latents=latents, peptides=np.array(peptides))
pd.DataFrame({'epoch': range(1, len(losses)+1), 'loss': losses}).to_csv('vae_loss.csv', index=False)

summary = {
    'model': 'ConvVAE', 'latent_dim': 64, 'n_peptides': len(peptides),
    'epochs': 80, 'initial_loss': float(losses[0]), 'final_loss': float(losses[-1]),
    'reconstruction_accuracy': float(np.mean(ids)),
    'latent_pca_variance': float(var), 'training_time_s': float(elapsed), 'device': str(device)
}
with open('training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('='*50)
print('TRAINING COMPLETE')
print('='*50)
for k, v in summary.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print('\nDownload these files (right sidebar -> Data -> download each):')
for f in ['conv_vae_weights.pt','vae_latents.npz','vae_loss.csv','training_summary.json','fig_s1_vae_loss.png','fig_s2_latent_pca.png']:
    print(f'  {f}')